# Create the datasets with EmbeddingGemma 2 embeddings

Builds drop-in replacements of `dataset_txt_only_False_embeddings_True_{train,val}.zip` in which **all embedding-derived variables** are recomputed from the EmbeddingGemma 2 product embeddings (`01_3_compute_embeddings_gemma.ipynb`) instead of the fine-tuned (DoubleMLDeep) embeddings:

* `emb_*`: centered and normalized embeddings (Matryoshka truncation to `emb_dim` dimensions, then L2 re-normalization)
* `pca_0`, ..., `pca_4`: first five principal components
* `similarity_cluster_0`, ..., `similarity_cluster_4`: cosine similarities to the k-means (k=5) centroids
* `neighbor_asin_*`, `neighbor_distance_*`, `weighted_substitute_price`: nearest-neighbour substitutes

The derived variables are computed with exactly the same functions and settings as in `01_1_create_dataset_txt_img.ipynb` (`center_and_norm` on the pooled train/val panel, `KMeans(n_clusters=5, n_init=200, random_state=42)`, `PCA(n_components=5, random_state=42)`, 5 nearest neighbours per date). All other variables (outcomes, prices, lags, controls, dummies) are taken unchanged from the original datasets, so that the estimation sample and all non-embedding variables are identical to the main analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

from utils.utils_data2 import (
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    compute_neighbors_and_distances,
    compute_neighbor_weighted_prices_lagged,
)
from utils.utils_gemma import load_embeddings

data_dir = "../data"
source_name = "dataset_txt_only_False_embeddings_True"
output_dir = "../output/01_4_create_dataset_gemma"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# (modality, Matryoshka dimension) combinations to build
configs = [
    ("txtimg", 256),  # main specification (same dimension as the fine-tuned embeddings)
    ("txt", 256),
    ("txtimg", 768),
]

## Original datasets

In [ ]:
df_orig_train = pd.read_csv(f"{data_dir}/{source_name}_train.zip")
df_orig_val = pd.read_csv(f"{data_dir}/{source_name}_val.zip")
num_train = len(df_orig_train)

emb_prefixes = (
    "emb_",
    "pca_",
    "similarity_cluster_",
    "neighbor_asin_",
    "neighbor_distance_",
)
embedding_derived_cols = [
    c for c in df_orig_train.columns if c.startswith(emb_prefixes)
] + ["weighted_substitute_price"]
other_cols = [c for c in df_orig_train.columns if c not in embedding_derived_cols]

print(f"train: {df_orig_train.shape}, val: {df_orig_val.shape}")
print(f"Embedding-derived columns replaced: {len(embedding_derived_cols)}")
print(f"Columns kept unchanged: {len(other_cols)}")

# pooled panel in the same order as in 01_1 (train rows first, then val rows)
df_full = pd.concat([df_orig_train, df_orig_val], axis=0, ignore_index=True)
df_full = df_full.set_index(["ASIN", "date"])
assert df_full.index.is_unique

# original (fine-tuned) cluster assignment, for comparison
sim_cols = [f"similarity_cluster_{i}" for i in range(5)]
orig_cluster = df_full[sim_cols].values.argmax(axis=1)

# column names used by the neighbour helpers in utils_data2
df_full_raw = df_full[["Q_t", "P_bb_t"]].rename(
    columns={"Q_t": "SALES_RANK", "P_bb_t": "BUYBOX_PRICE"}
)

## Build datasets

In [ ]:
def build_embedding_features(modality, emb_dim):
    # product-level embeddings, broadcast to the (ASIN, date) panel
    df_emb_products = load_embeddings(
        f"{data_dir}/embeddings_gemma/embeddinggemma2_{modality}.parquet", emb_dim=emb_dim
    )
    asins = df_full.index.get_level_values("ASIN")
    assert asins.isin(df_emb_products.index).all(), "Missing product embeddings"
    df_emb_rows = pd.DataFrame(
        df_emb_products.loc[asins].values,
        columns=range(emb_dim),
    )
    df_emb_rows.insert(0, "date", df_full.index.get_level_values("date"))
    df_emb_rows.insert(0, "ASIN", asins)

    # identical pipeline as in 01_1_create_dataset_txt_img.ipynb
    embeddings = center_and_norm(
        df_emb_rows.iloc[:num_train], df_emb_rows.iloc[num_train:]
    )
    print(f"Embedding shape: {embeddings.shape}")
    cluster_labels, cluster_centroids = get_cluster(embeddings, n_clusters=5, n_init=200)
    df_pca = get_pca(embeddings, n_components=5)
    df_sim = get_similarities(embeddings, cluster_centroids)

    df_features = df_pca.join(df_sim).join(embeddings.add_prefix("emb_"))

    # nearest-neighbour substitutes and weighted substitute price
    df_neighbors = df_full_raw.copy()
    df_neighbor_asins_by_date, df_distance_df_by_date = compute_neighbors_and_distances(
        embeddings, df_neighbors, n_neighbors=5
    )
    df_neighbors = df_neighbors.join(df_neighbor_asins_by_date)
    df_neighbors = df_neighbors.join(df_distance_df_by_date)
    weighted_substitute_price = compute_neighbor_weighted_prices_lagged(
        df_neighbors, "BUYBOX_PRICE"
    ).rename("weighted_substitute_price")
    df_features = df_features.join(
        df_neighbors.drop(columns=["SALES_RANK", "BUYBOX_PRICE"])
    ).join(weighted_substitute_price)
    return df_features.reindex(df_full.index), cluster_labels


def assemble(df_orig, df_features, emb_dim):
    df_orig = df_orig.set_index(["ASIN", "date"])
    df_feat = df_features.loc[df_orig.index]
    # keep the original column order, replace the embedding block by emb_0..emb_{emb_dim-1}
    columns = []
    for c in df_orig.columns:
        if c == "emb_0":
            columns.extend([f"emb_{i}" for i in range(emb_dim)])
        elif c.startswith("emb_"):
            continue
        else:
            columns.append(c)
    df_new = pd.concat(
        [df_orig[[c for c in other_cols if c not in ("ASIN", "date")]], df_feat], axis=1
    )[columns]
    return df_new.reset_index()

In [ ]:
cluster_assignments = {"finetuned": orig_cluster}
for modality, emb_dim in configs:
    dataframe_name = f"dataset_gemma_{modality}_dim{emb_dim}"
    print(f"\n===== {dataframe_name} =====")
    df_features, cluster_labels = build_embedding_features(modality, emb_dim)
    cluster_assignments[f"{modality}_dim{emb_dim}"] = df_features[sim_cols].values.argmax(axis=1)

    for split, df_orig in [("train", df_orig_train), ("val", df_orig_val)]:
        df_new = assemble(df_orig, df_features, emb_dim)
        # all non-embedding variables are unchanged and missing values occur in the same rows
        pd.testing.assert_frame_equal(df_new[other_cols], df_orig[other_cols])
        assert (
            df_new.isna().any(axis=1).values == df_orig.isna().any(axis=1).values
        ).all(), "Missing-value pattern changed"
        zip_args = dict(method="zip", archive_name=f"{dataframe_name}_{split}.csv")
        df_new.to_csv(
            f"{data_dir}/{dataframe_name}_{split}.zip", index=False, compression=zip_args
        )
        print(f"{split}: {df_new.shape} -> {data_dir}/{dataframe_name}_{split}.zip")

## Checks

The estimation samples (after `dropna()`) coincide with the samples of the main analysis (`main_train_keys.csv`, `main_val_keys.csv` written by `04_evaluation.ipynb`).

In [ ]:
for modality, emb_dim in configs:
    dataframe_name = f"dataset_gemma_{modality}_dim{emb_dim}"
    for split in ["train", "val"]:
        df_check = pd.read_csv(f"{data_dir}/{dataframe_name}_{split}.zip")
        df_check = df_check.drop(
            columns=["Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
                     "pred_ml_l_lag_1", "pred_ml_m_lag_1"]
        ).dropna()
        keys = pd.read_csv(f"main_{split}_keys.csv")
        same = df_check[["ASIN", "date"]].reset_index(drop=True).equals(keys)
        print(f"{dataframe_name} {split}: {df_check.shape}, same sample as main analysis: {same}")
        assert same

## Comparison of cluster assignments

In [ ]:
names = list(cluster_assignments)
ari = pd.DataFrame(
    [[adjusted_rand_score(cluster_assignments[a], cluster_assignments[b]) for b in names] for a in names],
    index=names,
    columns=names,
)
print("Adjusted Rand index between cluster assignments (all panel rows):")
ari.round(3)

In [ ]:
# Cross-tabulation of products: fine-tuned clusters (rows) vs. EmbeddingGemma 2 clusters (columns)
df_products = df_full[["subcat_aggregated"]].copy()
for name, labels in cluster_assignments.items():
    df_products[name] = labels
df_products = df_products.groupby(level="ASIN").first()
pd.crosstab(df_products["finetuned"], df_products["txtimg_dim256"], margins=True)

In [ ]:
# Cluster overview: size, dominant product types and the products closest to the centroid
texts = df_full["text"].groupby(level="ASIN").first()
overview = []
for name in names:
    for c in range(5):
        members = df_products[df_products[name] == c]
        top_cats = members["subcat_aggregated"].value_counts(normalize=True).head(3)
        overview.append(
            {
                "embedding": name,
                "cluster": c,
                "# Products": len(members),
                "Top product types": " / ".join(
                    f"{k} ({v:.0%})" for k, v in top_cats.items()
                ),
            }
        )
df_overview = pd.DataFrame(overview)
df_overview.to_csv(os.path.join(output_dir, "cluster_overview.csv"), index=False)
pd.set_option("display.max_colwidth", 200)
df_overview

In [ ]:
# Example products per cluster (main specification)
df_gemma_main = pd.read_csv(f"{data_dir}/dataset_gemma_txtimg_dim256_val.zip", usecols=["ASIN", "text"] + sim_cols)
df_gemma_main = df_gemma_main.groupby("ASIN").first()
for c in range(5):
    closest = df_gemma_main.sort_values(f"similarity_cluster_{c}", ascending=False).head(5)
    print(f"--- similarity_cluster_{c} ---")
    for asin, row in closest.iterrows():
        print(f"  {asin}: {row['text'][:110]}")